# Analyzing LLM Failures with Error Tracking

This notebook demonstrates how to use lab_llm's error tracking system to:
1. Track failures during LLM API calls
2. Analyze error patterns
3. Identify prompts that need fixes
4. Debug transient vs permanent failures

## Setup: Using ErrorTracker with LLMApi

The error tracker is optional but highly recommended for research workflows.

In [ ]:
import logging
from lab_llm.llm_api import LLMApi
from lab_llm.llm_cache import LLMCache
from lab_llm.duckdb_handler import DuckDBHandler
from lab_llm.error_callback_handler import ErrorCallbackHandler
from lab_llm.error_tracker import ErrorTracker
from lab_llm.constants import LLMModel, VersaOpenAi

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Create error tracker (JSONL file)
error_tracker = ErrorTracker("study_errors.jsonl")

# Create error handler with tracker
error_handler = ErrorCallbackHandler(logger, error_tracker=error_tracker)

# Setup cache and LLM API
db_handler = DuckDBHandler("cache.db")
cache = LLMCache(db_handler)
model = LLMModel(name=VersaOpenAi.GPT4_O_2024_08)

llm_api = LLMApi(
    cache=cache,
    seed=42,
    model_type=model,
    error_handler=error_handler,
    logging=logger,
    timeout=240,  # 4 minutes
    return_exceptions=True,  # Continue processing after errors
)

## Running Your Study

Process your prompts as normal. Errors will be automatically logged to the JSONL file.

In [ ]:
# Example: Process a batch of prompts
from lab_llm.dataset import TextDataset
import asyncio

prompts = [
    "Analyze this clinical note...",
    "Extract medications from...",
    # ... your prompts here
]

dataset = TextDataset(prompts)
results = await llm_api.get_outputs(dataset, batch_size=10)

# Check for None values (failures)
failures = [i for i, r in enumerate(results) if r is None]
print(f"Completed: {len(results) - len(failures)}/{len(results)}")
print(f"Failed: {len(failures)} prompts")

## Analyzing Errors

### 1. Quick Summary

In [ ]:
import pandas as pd

# Load error summary
summary = error_tracker.get_summary()
print("Error Summary:")
print(summary)

### 2. Transient Errors (Should Retry Automatically)

These are timeouts, rate limits, network errors. With the new caching policy, these will automatically retry on the next run.

In [ ]:
transient = error_tracker.get_transient_errors()

print(f"Total transient errors: {len(transient)}")
print("\nBreakdown by type:")
print(transient['error_type'].value_counts())

# Check if timeouts are too short
if 'timeout' in transient.columns:
    print(f"\nAverage timeout setting: {transient['timeout'].mean()}s")
    print("Consider increasing timeout if you see many TimeoutErrors")

### 3. Permanent Errors (Need Manual Fixes)

These are validation errors, serialization errors, etc. These prompts need to be fixed.

In [ ]:
permanent = error_tracker.get_permanent_errors()

print(f"Total permanent errors: {permanent['count'].sum()} across {len(permanent)} unique prompts")
print("\nTop 5 failing prompts:")

for idx, row in permanent.head(5).iterrows():
    print(f"\nPrompt Hash: {row['prompt_hash']}")
    print(f"  Error: {row['error_type']}")
    print(f"  Count: {row['count']}")
    print(f"  Message: {row['error_message'][:100]}...")
    if 'prompt_preview' in row:
        print(f"  Preview: {row['prompt_preview'][:150]}...")

### 4. Investigating Specific Failures

Drill down into errors for a specific prompt:

In [ ]:
# Get the prompt hash of a failing prompt
if not permanent.empty:
    problem_hash = permanent.iloc[0]['prompt_hash']
    
    # Get full error history for this prompt
    prompt_errors = error_tracker.get_errors_by_prompt(problem_hash)
    
    print(f"Error history for prompt {problem_hash}:")
    print(f"Total errors: {len(prompt_errors)}")
    print("\nTimeline:")
    
    for _, error in prompt_errors.iterrows():
        print(f"  [{error['timestamp']}] {error['error_type']}")
        print(f"    Category: {error['error_category']}")
        print(f"    Message: {error['error_message'][:100]}")

### 5. Advanced Analysis: Error Patterns Over Time

In [ ]:
# Load all errors
all_errors = error_tracker.load_errors()

if not all_errors.empty:
    # Convert timestamp to datetime
    all_errors['timestamp'] = pd.to_datetime(all_errors['timestamp'])
    
    # Plot errors over time (requires matplotlib)
    try:
        import matplotlib.pyplot as plt
        
        # Group by hour and category
        all_errors['hour'] = all_errors['timestamp'].dt.floor('H')
        hourly = all_errors.groupby(['hour', 'error_category']).size().unstack(fill_value=0)
        
        hourly.plot(kind='bar', stacked=True, figsize=(12, 6))
        plt.title('Errors Over Time by Category')
        plt.xlabel('Time')
        plt.ylabel('Error Count')
        plt.legend(title='Category')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("Install matplotlib for visualizations: pip install matplotlib")

### 6. Correlation with Cache

With the new caching policy (no None caching), you can check which prompts never succeeded:

In [ ]:
# Get unique failed prompt hashes
failed_hashes = set(all_errors['prompt_hash'].unique())

# Query cache to see which are cached (i.e., eventually succeeded)
import duckdb
conn = duckdb.connect("cache.db")
cached = conn.execute("SELECT DISTINCT prompt_hash FROM cache").fetchdf()
cached_hashes = set(cached['prompt_hash'].unique())
conn.close()

# Never succeeded = failed but not in cache
never_succeeded = failed_hashes - cached_hashes

print(f"Failed prompts: {len(failed_hashes)}")
print(f"Eventually succeeded: {len(failed_hashes & cached_hashes)}")
print(f"Never succeeded: {len(never_succeeded)}")

if never_succeeded:
    print("\nPrompts that never succeeded (need investigation):")
    for hash_val in list(never_succeeded)[:5]:
        errors = all_errors[all_errors['prompt_hash'] == hash_val]
        print(f"\n  {hash_val}")
        print(f"    Attempts: {len(errors)}")
        print(f"    Error types: {errors['error_type'].unique()}")
        if 'prompt_preview' in errors.iloc[0]:
            print(f"    Preview: {errors.iloc[0]['prompt_preview'][:100]}...")

## Next Steps

### For Transient Errors:
1. Re-run the same code - failures will retry automatically (not cached)
2. Consider increasing timeout if many TimeoutErrors
3. Check for rate limiting patterns

### For Permanent Errors:
1. Review the prompt previews
2. Fix validation issues in your Pydantic models
3. Update prompts that consistently fail
4. Re-run after fixes

### Clearing Error Logs:
If you've fixed issues and want a clean slate:

In [ ]:
# WARNING: This deletes all error logs
# error_tracker.clear()
# print("Error log cleared")